# 04 — Autoencoder Anomaly Detector

Trains a tabular autoencoder on **BENIGN traffic only**. At inference time the
reconstruction MSE is compared to a threshold (99th percentile on validation
benign) — anything above is flagged anomalous, possibly a zero-day attack the
MLP has never seen.


In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from src import config as cfg
from src.models import build_autoencoder, reconstruction_error
sns.set_theme(style="whitegrid")


## 1. Load benign-only training and full validation set

In [ ]:
train = pd.read_csv(cfg.PROCESSED_DIR / "train.csv")
val   = pd.read_csv(cfg.PROCESSED_DIR / "val.csv")

scaler = joblib.load(cfg.SCALER_FILE)
with open(cfg.FEATURE_NAMES_FILE) as f: feature_names = json.load(f)

train_benign = train[train[cfg.LABEL_COL] == "BENIGN"]
print("Benign train rows:", len(train_benign))

X_train = scaler.transform(train_benign[feature_names].astype(np.float32).values)
X_val   = scaler.transform(val[feature_names].astype(np.float32).values)
y_val_is_attack = (val[cfg.LABEL_COL] != "BENIGN").astype(int).values


## 2. Build the autoencoder

In [ ]:
ae = build_autoencoder(
    input_dim=X_train.shape[1],
    encoder_units=cfg.AE_CONFIG["encoder_units"],
    latent_dim=cfg.AE_CONFIG["latent_dim"],
    learning_rate=cfg.AE_CONFIG["learning_rate"],
)
ae.summary()


## 3. Train

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=cfg.AE_CONFIG["patience"],
                  restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]

# Train AE on benign only — internal val_split keeps things honest.
history = ae.fit(
    X_train, X_train,
    epochs=cfg.AE_CONFIG["epochs"],
    batch_size=cfg.AE_CONFIG["batch_size"],
    validation_split=0.1,
    callbacks=callbacks,
    verbose=2,
)


## 4. Choose a threshold from BENIGN validation reconstruction error

In [ ]:
val_benign = val[val[cfg.LABEL_COL] == "BENIGN"]
X_val_benign = scaler.transform(val_benign[feature_names].astype(np.float32).values)

err_benign_val = reconstruction_error(ae, X_val_benign)
threshold = float(np.percentile(err_benign_val, cfg.AE_CONFIG["threshold_percentile"]))
print(f"Threshold (P{cfg.AE_CONFIG['threshold_percentile']}):", threshold)


## 5. Visualise BENIGN vs ATTACK error histograms

In [ ]:
val_attack = val[val[cfg.LABEL_COL] != "BENIGN"]
X_val_attack = scaler.transform(val_attack[feature_names].astype(np.float32).values)
err_attack_val = reconstruction_error(ae, X_val_attack)

plt.figure(figsize=(10, 5))
sns.histplot(err_benign_val, bins=80, color="steelblue", label="BENIGN", stat="density")
sns.histplot(err_attack_val, bins=80, color="crimson",  label="ATTACK", stat="density",
             alpha=0.6)
plt.axvline(threshold, color="black", ls="--", label=f"threshold = {threshold:.4f}")
plt.xlabel("reconstruction MSE"); plt.title("Autoencoder anomaly score")
plt.legend(); plt.tight_layout(); plt.show()


## 6. Persist autoencoder and threshold

In [ ]:
cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)
ae.save(cfg.AE_MODEL_FILE)
with open(cfg.AE_THRESHOLD_FILE, "w") as f:
    json.dump({"threshold": threshold,
               "percentile": cfg.AE_CONFIG["threshold_percentile"]}, f, indent=2)

print("Saved AE   ->", cfg.AE_MODEL_FILE)
print("Saved thr  ->", cfg.AE_THRESHOLD_FILE)
